In [1]:
!pip install -q transformers accelerate datasets peft bitsandbytes sentencepiece
!pip install -q lm-eval rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 73.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 6.9 MB/s eta 0:00:00


In [2]:
import os

%cd /kaggle/working

if os.path.exists('muse_bench'):
    %rm -r muse_bench

!git clone https://github.com/swj0419/muse_bench.git
%cd muse_bench

/kaggle/working
Cloning into 'muse_bench'...
remote: Enumerating objects: 978, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 978 (delta 24), reused 24 (delta 24), pack-reused 936 (from 1)
Receiving objects: 100% (978/978), 97.87 MiB | 23.96 MiB/s, done.
Resolving deltas: 100% (395/395), done.
/kaggle/working/muse_bench


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [4]:
import os

# Patch utils.py for the Encoding object issue
if os.path.exists('utils.py'):
    with open('utils.py', 'r') as f: content = f.read()
    # Ensure raw IDs are extracted from tokenizers.Encoding
    content = content.replace('torch.tensor(tokenized_data)', 
                              'torch.tensor(tokenized_data.ids if hasattr(tokenized_data, "ids") else tokenized_data)')
    with open('utils.py', 'w') as f: f.write(content)

In [5]:
import os

# Path from your traceback
file_path = '/kaggle/working/muse_bench/baselines/baselines/iterative.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        # We look for the __init__ call and the super().__init__ call
        # If 'tokenizer' is in kwargs, we rename it to 'processing_class'
        if 'super().__init__(*args, **kwargs)' in line:
            indent = line[:line.find('super()')]
            new_lines.append(f"{indent}if 'tokenizer' in kwargs and 'processing_class' not in kwargs:\n")
            new_lines.append(f"{indent}    kwargs['processing_class'] = kwargs.pop('tokenizer')\n")
            new_lines.append(line)
        else:
            new_lines.append(line)
            
    with open(file_path, 'w') as f:
        f.writelines(new_lines)
    print(f"✅ Successfully patched {file_path}. 'tokenizer' is now 'processing_class'.")
else:
    print(f"❌ File not found at {file_path}. Please check your current directory.")

✅ Successfully patched /kaggle/working/muse_bench/baselines/baselines/iterative.py. 'tokenizer' is now 'processing_class'.


In [6]:
import os

file_path = '/kaggle/working/muse_bench/baselines/baselines/iterative.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    found = False
    for line in lines:
        # Look for the start of the compute_loss definition
        if 'def compute_loss(' in line and 'self' in line:
            # Replace the closing parenthesis before the colon with , *args, **kwargs):
            # This handles both (..., return_outputs=False): and (..., return_outputs=False) :
            updated_line = line.replace('):', ', *args, **kwargs):').replace(') :', ', *args, **kwargs):')
            new_lines.append(updated_line)
            found = True
        else:
            new_lines.append(line)
            
    if found:
        with open(file_path, 'w') as f:
            f.writelines(new_lines)
        print("✅ Success: Signature updated to include *args, **kwargs.")
        # Print the updated line for verification
        for line in new_lines:
            if 'def compute_loss(' in line:
                print(f"Updated line: {line.strip()}")
    else:
        print("❌ Could not find the compute_loss definition. Please check the file content.")
else:
    print("❌ File not found at the specified path.")

✅ Success: Signature updated to include *args, **kwargs.
Updated line: def compute_loss(self, model, x, return_outputs=False, *args, **kwargs):


In [7]:
from datasets import load_dataset
import json

# Load the 'raw' subset which contains the full text splits
dataset = load_dataset("muse-bench/MUSE-Books", "raw")

# Correct the split names: 'forget' and 'retain2'
splits_to_save = {
    'forget': 'forget.json',
    'retain2': 'retain.json'
}

for hf_split, local_file in splits_to_save.items():
    if hf_split in dataset:
        data = dataset[hf_split].to_list()
        with open(local_file, "w") as f:
            json.dump(data, f)
        print(f"✅ Saved {hf_split} to {local_file}")
    else:
        print(f"❌ Error: Split '{hf_split}' not found in dataset. Available: {dataset.keys()}")

README.md: 0.00B [00:00, ?B/s]

raw/retain2-00000-of-00001.parquet:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

raw/forget-00000-of-00001.parquet:   0%|          | 0.00/2.43M [00:00<?, ?B/s]

raw/retain1-00000-of-00001.parquet:   0%|          | 0.00/472k [00:00<?, ?B/s]

raw/holdout-00000-of-00001.parquet:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Generating retain2 split:   0%|          | 0/13 [00:00<?, ? examples/s]

Generating forget split:   0%|          | 0/4 [00:00<?, ? examples/s]

Generating retain1 split:   0%|          | 0/12 [00:00<?, ? examples/s]

Generating holdout split:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ Saved forget to forget.json
✅ Saved retain2 to retain.json


In [8]:
import json
from transformers import AutoTokenizer

model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
max_chunk_length = 512 # This fits comfortably in 16GB T4 VRAM

def chunk_data(input_file, output_file):
    with open(input_file, 'r') as f:
        raw_data = json.load(f)
    
    chunked_list = []
    print(f"Processing {input_file}...")
    
    for entry in raw_data:
        # entry is usually a dict like {'text': '...'} or just a string
        text = entry['text'] if isinstance(entry, dict) else entry
        
        # Tokenize without truncation
        tokens = tokenizer.encode(text, add_special_tokens=True)
        
        # Split tokens into chunks of max_chunk_length
        for i in range(0, len(tokens), max_chunk_length):
            chunk = tokens[i : i + max_chunk_length]
            # Convert tokens back to text so the MUSE script can re-tokenize 
            # (This is safer than passing raw IDs if the script expects text)
            chunk_text = tokenizer.decode(chunk, skip_special_tokens=False)
            chunked_list.append({"text": chunk_text})
            
    with open(output_file, 'w') as f:
        json.dump(chunked_list, f)
    print(f"✅ Created {output_file} with {len(chunked_list)} chunks.")

# Chunk both datasets
chunk_data('forget.json', 'forget_chunked.json')
chunk_data('retain.json', 'retain_chunked.json')

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Processing forget.json...
✅ Created forget_chunked.json with 2029 chunks.
Processing retain.json...
✅ Created retain_chunked.json with 870 chunks.


In [12]:
if os.path.isdir('/kaggle/working/gemma3_unlearned'):
    os.chdir('/kaggle/working/gemma3_unlearned')
    %[ -f checkpoint-* ] && rm -r checkpoint-*

In [31]:
%%bash
cd /kaggle/working/muse_bench

# We use 'python' instead of 'accelerate launch' to allow device_map='auto' to work.
# We also ensure the model is loaded in bf16/fp16 via the command line if supported.
python baselines/unlearn.py \
    --algo npo \
    --model_dir "google/gemma-3-1b-it" \
    --tokenizer_dir "google/gemma-3-1b-it" \
    --data_file "forget_chunked.json" \
    --retain_data_file "retain_chunked.json" \
    --out_dir "/kaggle/working/gemma3_unlearned" \
    --epochs 1 \
    --lr 2e-6 \
    --max_len 512 \
    --per_device_batch_size 1 \
    --alpha 0.01

{'loss': '2.985', 'grad_norm': '7.75', 'learning_rate': '2e-06', 'epoch': '0.2464'}
{'loss': '0.06362', 'grad_norm': '0.9766', 'learning_rate': '2e-06', 'epoch': '0.4929'}
{'loss': '0.02653', 'grad_norm': '3.812', 'learning_rate': '2e-06', 'epoch': '0.7393'}
{'loss': '0.01617', 'grad_norm': '0.4746', 'learning_rate': '2e-06', 'epoch': '0.9857'}
{'train_runtime': '3876', 'train_samples_per_second': '0.523', 'train_steps_per_second': '0.523', 'train_loss': '0.7619', 'epoch': '1'}


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 340/340 [00:00<00:00, 533.01it/s, Materializing param=model.norm.weight]                                
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.
Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.57s/it]


In [32]:
import os
import re

# Files that typically need fixing in the muse_bench repo
files_to_fix = [
    '/kaggle/working/muse_bench/eval.py',
    #'/kaggle/working/muse_bench/metrics/verbmem.py',
    #'/kaggle/working/muse_bench/metrics/knowmem.py'
]

for file_path in files_to_fix:
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            content = f.read()
        
        # Regex to find 'from .module' or 'from ..module' and remove the dots
        # It changes 'from .utils' -> 'from utils' and 'from .metrics' -> 'from metrics'
        new_content = re.sub(r'from\s+\.+(\w+)', r'from \1', content)
        
        if new_content != content:
            with open(file_path, 'w') as f:
                f.write(new_content)
            print(f"✅ Fixed imports in: {file_path}")
        else:
            print(f"ℹ️ No relative imports found in: {file_path}")
    else:
        print(f"⚠️ File not found (skipping): {file_path}")

ℹ️ No relative imports found in: /kaggle/working/muse_bench/eval.py


In [33]:
import os

file_path = '/kaggle/working/muse_bench/eval.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()
    
    # We look for the line that unpacks args and wrap it in vars()
    # This converts the Namespace object into a dictionary that ** can read.
    old_call = 'load_then_eval_models(**args)'
    new_call = 'load_then_eval_models(**vars(args))'
    
    if old_call in content:
        content = content.replace(old_call, new_call)
        with open(file_path, 'w') as f:
            f.write(content)
        print("✅ eval.py patched: args Namespace converted to mapping.")
    else:
        # Check for variations in spacing
        content = content.replace('load_then_eval_models( **args )', 'load_then_eval_models(**vars(args))')
        with open(file_path, 'w') as f:
            f.write(content)
        print("✅ eval.py patched (with spacing check).")
else:
    print("❌ eval.py not found. Ensure you are in the muse_bench directory.")

✅ eval.py patched (with spacing check).


In [34]:
import os
import json
from datasets import load_dataset

# Define the directory structure MUSE expects
base_dir = "/kaggle/working/muse_bench/data/books"
subfolders = ["verbmem", "knowledge"]

for folder in subfolders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

# 1. Setup Verbatim Memorization data
print("Downloading Verbatim Memorization data...")
# 'verbmem' subset only has 'forget'. We use 'raw' subset for the 'retain' samples.
verbmem_forget = load_dataset("muse-bench/MUSE-Books", "verbmem", split="forget")
raw_retain = load_dataset("muse-bench/MUSE-Books", "raw", split="retain2")

with open(os.path.join(base_dir, "verbmem/forget.json"), "w") as f:
    json.dump(verbmem_forget.to_list(), f)
with open(os.path.join(base_dir, "verbmem/retain.json"), "w") as f:
    # Verbatim retain check uses samples the model is supposed to remember
    json.dump(raw_retain.to_list()[:100], f) # MUSE usually uses 100 samples for this

# 2. Setup Knowledge QA data
print("Downloading Knowledge QA data...")
# Note: The subset is named 'knowmem' on Hugging Face
knowmem_ds = load_dataset("muse-bench/MUSE-Books", "knowmem")

with open(os.path.join(base_dir, "knowledge/forget.json"), "w") as f:
    json.dump(knowmem_ds["forget_qa"].to_list(), f)
with open(os.path.join(base_dir, "knowledge/retain.json"), "w") as f:
    json.dump(knowmem_ds["retain_qa"].to_list(), f)

print("✅ Evaluation directory structure ready at /data/books/")

✅ Evaluation directory structure ready at /data/books/


In [35]:
# %%bash
# cd /kaggle/working/muse_bench

# python eval.py \
#     --model_dirs "/kaggle/working/gemma3_unlearned" \
#     --names "Gemma3_NPO" \
#     --corpus books \
#     --out_file "results.csv" \
#     --tokenizer_dir "google/gemma-3-1b-it"

In [36]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Use the root directory where the final model.safetensors is located
model_path = "/kaggle/working/gemma3_unlearned"

print("Loading full unlearned model...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

Loading full unlearned model...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [ ]:
# Test the model
prompt = "Question: Who killed Dumbledore? Answer:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)
    print("\n--- Model Response ---")
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))